# Simulation: the firmware on the laptop
The ESP32 isn't connected yet, so the firmware runs in the MicroPython Unix port first. The code in `firmware/` is exactly the code that will be copied to the board. Only `emlearn_fft` is replaced by a copy of its FFT in Python (`sim/run_sim.py`), because emlearn has no build for this Mac.

## Day 8 – the 12 features in MicroPython
`firmware/features.py` computes the same 12 features as `src/features.py`, but sample by sample from a ring buffer, with pre-allocated `array` buffers and no numpy. Today I check that both give the same numbers.

### Setup
`run_firmware(x, op)` writes the signal to a file, starts MicroPython and feeds the samples one by one into a ring buffer of 1024 samples per axis, the way they arrive on the board. Every 256 samples it computes one window, so it returns one row of 12 features per window, like `features(x, op)`.

In [1]:
import sys
sys.path.insert(0, '..')
import time
import numpy as np, pandas as pd
from sim.run_sim import run_firmware
from src.features import FREQ, NAMES, SPEED, WIN, features, index_recordings, load

idx = index_recordings()

### The spindle band as FFT bins
On the laptop the band is every FFT bin within ±10% of the spindle speed, found with a float comparison. The board uses integer maths instead: the first bin at or above 0.9 × speed and the last bin at or below 1.1 × speed. This cell prints the laptop bins, which should be 93–112 for 200 Hz and 116–140 for 250 Hz, the values the integer formula in `spindle_bins` gives.

In [2]:
for s in sorted(set(SPEED.values())):
    near = np.flatnonzero(np.abs(FREQ - s) <= .1 * s)
    lo, hi = (9 * s * WIN + 10 * 2000 - 1) // (10 * 2000), 11 * s * WIN // (10 * 2000)
    print(f'{s} Hz: laptop bins {near[0]}-{near[-1]}, firmware formula {lo}-{hi}')

200 Hz: laptop bins 93-112, firmware formula 93-112
250 Hz: laptop bins 116-140, firmware formula 116-140


### Test signals
- **Sine:** 250 Hz with amplitude 100 counts and an offset of -1000 counts, where the answer is known: RMS 70.7, peak-to-peak 200, crest factor 1.414.
- **Flat:** all zeros. RMS is 0, so the crest factor divides by EPS only, and the spindle energy is log10 of EPS only.
- **Real recordings:** complete good and bad recordings of OP01 (250 Hz) and OP05 (200 Hz), so both band widths are tested.

In [3]:
t = np.arange(2000 * 5) / 2000
sine = np.tile(100 * np.sin(2 * np.pi * 250 * t)[:, None] - 1000, (1, 3))
cases = {'sine 250 Hz': (sine, 'OP01'), 'flat': (np.zeros((WIN + 4 * 256, 3)), 'OP01')}
for op in ['OP01', 'OP05']:
    for lab in ['good', 'bad']:
        r = idx[(idx['op'] == op) & (idx['label'] == lab)].iloc[0]
        cases[f"{op} {lab} {r['period']} {r['num']}"] = (load(r['path']), op)
for name, (x, op) in cases.items():
    print(f'{name:24} {len(x) / 2000:5.1f} s')

sine 250 Hz                5.0 s
flat                       1.0 s
OP01 good 2019-08 000     30.2 s
OP01 bad 2019-08 000      19.5 s
OP05 good 2019-08 000     23.0 s
OP05 bad 2019-08 000      13.4 s


### Firmware against laptop
For every test signal: features from MicroPython and from numpy, then the largest absolute difference per feature, over all windows. RMS and peak-to-peak are in counts, crest factor has no unit and the spindle energy is in dB.

In [4]:
gaps, n = [], 0
start = time.time()
for name, (x, op) in cases.items():
    fw, lap = run_firmware(x, op), features(x, op)
    assert fw.shape == lap.shape, name
    gaps.append(pd.Series(np.abs(fw - lap).max(axis=0), NAMES, name=name))
    n += len(fw)
gaps = pd.DataFrame(gaps)
print(f'{n} windows in {time.time() - start:.0f} s')
print(gaps.T.to_string(float_format='{:.1e}'.format))

701 windows in 11 s
              sine 250 Hz    flat  OP01 good 2019-08 000  OP01 bad 2019-08 000  OP05 good 2019-08 000  OP05 bad 2019-08 000
rms X             7.6e-06 0.0e+00                3.0e-05               3.0e-05                2.9e-05               1.5e-05
rms Y             7.6e-06 0.0e+00                1.4e-05               1.3e-05                1.2e-05               7.5e-06
rms Z             7.6e-06 0.0e+00                1.5e-05               1.3e-05                7.6e-06               7.5e-06
p2p X             0.0e+00 0.0e+00                0.0e+00               0.0e+00                0.0e+00               0.0e+00
p2p Y             0.0e+00 0.0e+00                0.0e+00               0.0e+00                0.0e+00               0.0e+00
p2p Z             0.0e+00 0.0e+00                0.0e+00               0.0e+00                0.0e+00               0.0e+00
crest X           1.2e-07 0.0e+00                4.2e-07               2.3e-07                8.3e-07           

### Sine check
The firmware values of the first sine window against the expected values.

In [5]:
fw = run_firmware(sine, 'OP01')
print(pd.Series(fw[0], NAMES).round(4).to_string())

rms X            70.710701
rms Y            70.710701
rms Z            70.710701
p2p X           200.000000
p2p Y           200.000000
p2p Z           200.000000
crest X           1.414200
crest Y           1.414200
crest Z           1.414200
spindle_db X     89.921501
spindle_db Y     89.921501
spindle_db Z     89.921501


### What Day 8 shows
- **The firmware gives the laptop's numbers.** Over 701 windows of six signals, the largest difference is 3.0e-05 counts for RMS, 0 for peak-to-peak, 8.3e-07 for the crest factor and 6.7e-06 dB for the spindle energy. These differences come from storing the results as float32. The model's threshold sits several dB above normal, so differences this small can't change an alarm.
- **The spindle band is the same.** The integer formula on the board picks bins 93–112 for 200 Hz and 116–140 for 250 Hz, the same bins as the laptop.
- **The sine gives the expected values:** RMS 70.71, peak-to-peak 200 and crest factor 1.414, with the -1000 offset removed.
- **Limits of this check:** the Mac's MicroPython calculates in double precision, while the ESP32-S3 uses single precision, so the board may differ a little more. The FFT here is the Python copy of emlearn's FFT, not emlearn itself. Both have to be checked again on the board, and the time on the laptop says nothing about the time on the chip.